# Exercícios

In [1]:
import pandas as pd

df = pd.read_csv("https://dados-ml-pln.s3.sa-east-1.amazonaws.com/tweets_classificados.csv", encoding='utf-8')
df.head()

,id,data_tweet,texto,sentimento
0,0,Sun Jan 08 01:22:05 +0000 2017,���⛪ @ Catedral de Santo Antônio - Governador ...,Neutro
1,1,Sun Jan 08 01:49:01 +0000 2017,"� @ Governador Valadares, Minas Gerais https:/...",Neutro
2,2,Sun Jan 08 01:01:46 +0000 2017,"�� @ Governador Valadares, Minas Gerais https:...",Neutro
3,3,Wed Jan 04 21:43:51 +0000 2017,��� https://t.co/BnDsO34qK0,Neutro
4,4,Mon Jan 09 15:08:21 +0000 2017,��� PSOL vai questionar aumento de vereadores ...,Negativo


## ToDo 1

Altere as funções de tratamento de texto apresentadas em sala para que elas façam a remoção de links também. 

Crie uma nova coluna chamada texto_tratado que conterá o resultado da aplicação das funções. 

In [2]:
# resposta
# criar funções de pré-processamento de texto que incluam remoção de links, e aplicar no DataFrame

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

def remover_links(texto):
    return re.sub(r'http\S+|www\S+', '', texto)

def remover_caracteres_especiais(texto):
    return re.sub(r'[^a-zA-ZÀ-ÿ\s]', '', texto)

def remover_stopwords(texto):
    stop_words = set(stopwords.words('portuguese'))
    tokens = word_tokenize(texto, language='portuguese')
    tokens_filtrados = [t for t in tokens if t.lower() not in stop_words]
    return ' '.join(tokens_filtrados)

def tratar_texto(texto):
    texto = remover_links(texto)
    texto = remover_caracteres_especiais(texto)
    texto = texto.lower()
    texto = remover_stopwords(texto)
    return texto

df['texto_tratado'] = df['texto'].apply(tratar_texto)

df[['texto', 'texto_tratado']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tatia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tatia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tatia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,texto,texto_tratado
0,���⛪ @ Catedral de Santo Antônio - Governador ...,catedral santo antônio governador valadaresmg
1,"� @ Governador Valadares, Minas Gerais https:/...",governador valadares minas gerais
2,"�� @ Governador Valadares, Minas Gerais https:...",governador valadares minas gerais
3,��� https://t.co/BnDsO34qK0,
4,��� PSOL vai questionar aumento de vereadores ...,psol vai questionar aumento vereadores prefeit...


O que cada função faz:

remover_links → remove URLs com http, https ou www usando regex

remover_caracteres_especiais → remove emojis, pontuação e símbolos (mantém letras com acento)

remover_stopwords → remove palavras sem valor semântico como "de", "que", "para"

tratar_texto → orquestra tudo em ordem, incluindo o .lower() para deixar tudo minúsculo



A coluna texto_tratado vai aparecer limpa ao lado da original para você comparar.

## ToDo 2

Ao fazer a remoção de links, percebemos que algumas linhas da coluna texto_tratado possuem valores faltantes. Entretanto, o Python trata eles como ''(str) e nao como Null. Assim, um simples dropna nao resolve o problema. 

Encontre uma forma de remover tais elementos.

In [3]:
#resposta
#Para remover as linhas onde o texto tratado ficou vazio após o processamento:

# Substitui strings vazias ou só com espaços por NaN, depois remove
df['texto_tratado'] = df['texto_tratado'].str.strip()
df = df[df['texto_tratado'] != '']
df = df.reset_index(drop=True)

df.shape  # para verificar quantas linhas sobraram

(5763, 5)

O que foi feito:

.str.strip() → remove espaços extras que possam ter sobrado
df[df['texto_tratado'] != ''] → filtra fora as linhas com string vazia
reset_index → reorganiza os índices após a remoção

In [4]:
# Se quiser confirmar que não sobrou nenhum vazio:
print((df['texto_tratado'] == '').sum())  # deve retornar 0

0


## ToDo 3

Separe a coluna texto_tratado em conjunto de treino e teste na proporção 70/30

In [5]:
#resposta
from sklearn.model_selection import train_test_split

X = df['texto_tratado']
y = df['sentimento']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")

Treino: 4034 amostras
Teste:  1729 amostras


X → texto (features)
y → sentimento (target)
test_size=0.3 → 30% para teste, 70% para treino
random_state=42 → garante que a divisão seja reproduzível toda vez que rodar

## ToDo 4

Transforme os dados para criar a representação numérica dos textos. Use uma versão com CountVectorizer e outra com TFIDFVectorizer

In [6]:
# resposta - CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# CountVectorizer
count_vect = CountVectorizer()
X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

print(f"CountVectorizer - Treino: {X_train_count.shape} | Teste: {X_test_count.shape}")


CountVectorizer - Treino: (4034, 5614) | Teste: (1729, 5614)


In [7]:
# resposta - TFIDFVectorizer
# TfidfVectorizer
tfidf_vect = TfidfVectorizer()
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

print(f"TfidfVectorizer - Treino: {X_train_tfidf.shape} | Teste: {X_test_tfidf.shape}")

TfidfVectorizer - Treino: (4034, 5614) | Teste: (1729, 5614)


Pontos importantes:

.fit_transform() apenas no treino → o vocabulário é aprendido só com os dados de treino

.transform() no teste → aplica o mesmo vocabulário aprendido, sem "vazar" informação do teste para o treino

CountVectorizer → conta a frequência bruta de cada palavra

TfidfVectorizer → pondera a frequência pela importância da palavra no corpus (penaliza palavras muito comuns)

## ToDo 5

Treine um modelo SVM nas duas abordagens e compare seus resultados

In [11]:
# resposta - CountVecorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Modelo com CountVectorizer
svm_count = SVC(kernel='linear', random_state=42)
svm_count.fit(X_train_count, y_train)
y_pred_count = svm_count.predict(X_test_count)



In [12]:
# resposta - TFIDFVectorizer
# Modelo com TfidfVectorizer
svm_tfidf = SVC(kernel='linear', random_state=42)
svm_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = svm_tfidf.predict(X_test_tfidf)

In [13]:
# Comparação
print("=== SVM + CountVectorizer ===")
print(classification_report(y_test, y_pred_count))

print("=== SVM + TfidfVectorizer ===")
print(classification_report(y_test, y_pred_tfidf))

=== SVM + CountVectorizer ===
              precision    recall  f1-score   support

    Negativo       0.91      0.88      0.89       299
      Neutro       0.92      0.94      0.93       592
    Positivo       0.98      0.98      0.98       838

    accuracy                           0.95      1729
   macro avg       0.94      0.93      0.93      1729
weighted avg       0.95      0.95      0.95      1729

=== SVM + TfidfVectorizer ===
              precision    recall  f1-score   support

    Negativo       0.92      0.88      0.90       299
      Neutro       0.92      0.95      0.93       592
    Positivo       0.98      0.97      0.98       838

    accuracy                           0.95      1729
   macro avg       0.94      0.93      0.94      1729
weighted avg       0.95      0.95      0.95      1729



O classification_report mostra para cada classe (Positivo, Negativo, Neutro):

precision → do que o modelo disse ser X, quanto realmente era X

recall → do que realmente era X, quanto o modelo acertou

f1-score → média harmônica entre precision e recall (métrica mais equilibrada)

accuracy → percentual geral de acertos

Em geral o TF-IDF tende a performar melhor em tarefas de texto porque penaliza palavras muito frequentes que carregam pouco significado, mas vale comparar os resultados com seus dados para confirmar.

## ToDo 6
Crie uma função que lematiza as palavras da coluna texto_tratado apenas se elas forem um verbo. Depois, crie uma nova coluna chamada texto_tratado_lemma que conterá o resultado da aplicação da função na coluna texto_tratado. 

Dica: use o Corpus pt_core_news_sm como referência para determinar a classe gramatical da palavra

In [ ]:
#%pip install spacy
#%python -m spacy download pt_core_news_sm

UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


In [16]:
# resposta
import spacy

nlp = spacy.load('pt_core_news_sm')

def lematizar_verbos(texto):
    doc = nlp(texto)
    tokens = [token.lemma_ if token.pos_ == 'VERB' else token.text for token in doc]
    return ' '.join(tokens)

df['texto_tratado_lemma'] = df['texto_tratado'].apply(lematizar_verbos)

df[['texto_tratado', 'texto_tratado_lemma']].head(10)

,texto_tratado,texto_tratado_lemma
0,catedral santo antônio governador valadaresmg,catedral santo Antônio governador valadaresmg
1,governador valadares minas gerais,governador valadares minas gerais
2,governador valadares minas gerais,governador valadares minas gerais
3,psol vai questionar aumento vereadores prefeit...,psol vai questionar aumento vereadores prefeit...
4,bom bandido morto deputado cabo júlio condenad...,bom bandido matar deputado cabo júlio condenar...
5,mineiros dizem torcer time nenhummesmo dentro ...,mineiros dizer torcer time nenhummesmo dentro ...
6,gigantesca barba mal destaque caderno cultura ...,gigantesca barbar mal destaque caderno cultura...
7,bb governo minas travam disputa sobre depósito...,bb governo minas trar disputa sobre depósitos ...
8,vcs bh fica pequena belo horizonte pron bloizõ...,vcsr bh ficar pequena belo horizonte pron bloi...
9,daí gente visita governador valadares lugar eh...,daí gente visitar governador valadares lugar e...


O que a função faz token por token:

token.pos_ == 'VERB' → verifica se a palavra é um verbo
token.lemma_ → se for verbo, substitui pela forma base (ex: "conectamos" → "conectar")
token.text → se não for verbo, mantém a palavra original

Exemplo do resultado:
texto_tratadotexto_tratado_lemmaestamos correndo risco chuvaestar correr risco chuvacidade bonita pessoas felizescidade bonita pessoas felizes
Só os verbos são lematizados, o resto das palavras permanece intacto.

## ToDo 7

repita os ToDo 3, ToDo 4 e ToDo 5, usando como feature a coluna texto_tratado_lemma, e veja se os resultados tiveram melhora.

In [17]:
#resposta
# ToDo 3 - Split treino/teste com texto_tratado_lemma
X_lemma = df['texto_tratado_lemma']
y_lemma = df['sentimento']

X_train_lemma, X_test_lemma, y_train_lemma, y_test_lemma = train_test_split(
    X_lemma, y_lemma, test_size=0.3, random_state=42
)

print(f"Treino: {X_train_lemma.shape[0]} amostras")
print(f"Teste:  {X_test_lemma.shape[0]} amostras")

Treino: 4034 amostras
Teste:  1729 amostras


In [18]:
# ToDo 4 - Vetorização
count_vect_lemma = CountVectorizer()
X_train_count_lemma = count_vect_lemma.fit_transform(X_train_lemma)
X_test_count_lemma = count_vect_lemma.transform(X_test_lemma)

tfidf_vect_lemma = TfidfVectorizer()
X_train_tfidf_lemma = tfidf_vect_lemma.fit_transform(X_train_lemma)
X_test_tfidf_lemma = tfidf_vect_lemma.transform(X_test_lemma)

print(f"CountVectorizer - Treino: {X_train_count_lemma.shape} | Teste: {X_test_count_lemma.shape}")
print(f"TfidfVectorizer - Treino: {X_train_tfidf_lemma.shape} | Teste: {X_test_tfidf_lemma.shape}")

CountVectorizer - Treino: (4034, 5333) | Teste: (1729, 5333)
TfidfVectorizer - Treino: (4034, 5333) | Teste: (1729, 5333)


In [19]:
# ToDo 5 - Treino e comparação dos SVMs
svm_count_lemma = SVC(kernel='linear', random_state=42)
svm_count_lemma.fit(X_train_count_lemma, y_train_lemma)
y_pred_count_lemma = svm_count_lemma.predict(X_test_count_lemma)

svm_tfidf_lemma = SVC(kernel='linear', random_state=42)
svm_tfidf_lemma.fit(X_train_tfidf_lemma, y_train_lemma)
y_pred_tfidf_lemma = svm_tfidf_lemma.predict(X_test_tfidf_lemma)

print("=== SVM + CountVectorizer + Lematização ===")
print(classification_report(y_test_lemma, y_pred_count_lemma))

print("=== SVM + TfidfVectorizer + Lematização ===")
print(classification_report(y_test_lemma, y_pred_tfidf_lemma))

=== SVM + CountVectorizer + Lematização ===
              precision    recall  f1-score   support

    Negativo       0.91      0.88      0.89       299
      Neutro       0.92      0.94      0.93       592
    Positivo       0.98      0.97      0.98       838

    accuracy                           0.95      1729
   macro avg       0.94      0.93      0.93      1729
weighted avg       0.95      0.95      0.95      1729

=== SVM + TfidfVectorizer + Lematização ===
              precision    recall  f1-score   support

    Negativo       0.92      0.88      0.90       299
      Neutro       0.92      0.95      0.93       592
    Positivo       0.98      0.98      0.98       838

    accuracy                           0.95      1729
   macro avg       0.94      0.93      0.94      1729
weighted avg       0.95      0.95      0.95      1729



Compare o f1-score de cada abordagem:
AbordagemEsperadoCount sem lemabaselineTF-IDF sem lemageralmente melhor que CountCount + lemapode melhorar ao reduzir variações verbaisTF-IDF + lematende a ser a melhor combinação
A lematização ajuda quando há muitas variações do mesmo verbo no corpus (ex: "fui", "foi", "fomos" → todos viram "ir"), reduzindo o vocabulário e melhorando a generalização do modelo.

In [ ]:
#resposta - CountVectorizer

In [ ]:
# resposta - TFIDFVectorizer